# Chapter 9: LLMOps and Deployment

Estimated time: ~6 hours.

Prerequisites: Chapter 4 (reliability patterns), Chapter 5 (cost/latency), Chapter 8 (this
chapter is where "how will you know it's working, in production, over time," question 8 of
Chapter 8's framework, actually gets answered).

## Concept: what's actually different about deploying an LLM/agent system

Traditional software deployment already solved "ship code safely": canary releases, feature
flags, rollback. LLMOps borrows all of it, plus one thing that doesn't have a
classical-software equivalent. The prompt is a deployable artifact with its own version
history, exactly as consequential as code, and it changes far more often than code does.

#### Prompt versioning

A prompt edited in place, with no record of what changed or when, is this chapter's version
of Chapter 4's stale-cache lesson: the failure isn't visible until something goes wrong, and
by then there's no record of what the "before" state even was. Treat every prompt change as
a new, immutable version, never an edit, with a pointer (`current_version`) that gets moved,
not a value that gets overwritten. This is what makes rollback possible at all: rolling back
means moving the pointer, not trying to reconstruct what the prompt used to say.

#### Canary releases and progressive delivery

Route a small percentage of real traffic to a new version, watch it, and ramp up only if it
looks healthy, the same pattern web services use for code deploys, applied to a prompt or
model swap. The stakes are different, though. A bad code deploy usually fails loudly (a 500
error, a crash); a bad prompt change often fails quietly: slightly worse answers, a subtly
higher refusal rate, nothing that trips an infrastructure alert. This is why LLMOps needs
its own metrics, not just infrastructure health checks.

#### Shadow deployment

Run the new version on real traffic without serving its output to users. Log what it would
have said, compare against the current version offline, and only promote to a real canary
once the shadow comparison looks good. This is strictly safer than a canary for a high-stakes
change, at the cost of not measuring real user reactions (only offline comparison) until the
canary stage actually starts.

#### Drift detection

Two distinct kinds are worth naming separately. Behavioral drift happens in your own system:
the new prompt/model version's output distribution has shifted, and this chapter's build
section detects this directly. Upstream drift is different: the underlying provider silently
updates a model version behind a stable-sounding name, and your system's behavior shifts
without you having changed anything at all. It's a real, recurring failure mode with hosted
LLM APIs that has no classical-software equivalent.

#### Rollback

Rollback is only as fast as your slowest cache. A rollback that flips `current_version` but
leaves a stale cached response, or a cached "current version" pointer read once at startup,
isn't a rollback at all. It's a rollback that hasn't taken effect yet. This chapter's
break-it section makes this concrete.

## Setup

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random

from agentlib.grading import check
from agentlib import llm_client

random.seed(42)
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print("This chapter makes no model calls: responses come from a deterministic simulator so\n"
      "a canary comparison is reproducible run to run, which a real model would not be.\n"
      "HAS_KEY is printed for consistency with the other chapters, not because anything\n"
      "here branches on it.")

LLM_PROVIDER = 'anthropic', HAS_KEY = False
This chapter makes no model calls: responses come from a deterministic simulator so
a canary comparison is reproducible run to run, which a real model would not be.
HAS_KEY is printed for consistency with the other chapters, not because anything
here branches on it.


## Build: an immutable prompt registry and a canary router

`PromptRegistry` enforces immutability directly in code, not just as a convention someone
has to remember: `publish()` refuses to overwrite an existing version ID, and moving
`current_version` (via `promote()`) is the only way to change what's live. Rollback is then
just "promote an older version," not a special operation.

The registry is yours to build, and "immutable" is the load-bearing word. A store that lets
you edit a published version passes every test that writes a prompt and reads it straight
back. It fails exactly once, in production, on the day you try to roll back -- because the
pointer moves and the text it points at was overwritten weeks ago. That is break-it #1 below,
and building the registry properly is what makes it impossible.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PromptVersion:
    '''frozen=True is not decoration. It is what stops a published prompt being edited in
    place, which is what makes the audit trail describe prompts rather than just ids.'''
    version_id: str
    text: str


class PromptRegistry:
    '''An append-only store of prompt versions, plus a pointer at whichever one is live.

    - publish(version_id, text) -> PromptVersion. Store it and return it. Raise ValueError
      if that id already exists: versions are immutable, so a change means a new id.
    - promote(version_id) -> None. Point current_version at it and append it to history.
      Raise ValueError if the id was never published.
    - get(version_id) -> PromptVersion.

    Publishing is not promoting. A new version goes on the shelf; promoting is what points
    traffic at it, and conflating the two makes every publish a deploy.

    history is the audit trail, so append on EVERY promote -- including a promote back to a
    version that ran before. That repeat is the rollback, and it is the entry an incident
    review goes looking for.
    '''

    def __init__(self):
        self._versions: dict[str, PromptVersion] = {}
        self.current_version: str | None = None
        self.history: list[str] = []

    def publish(self, version_id: str, text: str) -> PromptVersion:
        raise NotImplementedError("Implement me, then re-run this cell")

    def promote(self, version_id: str) -> None:
        raise NotImplementedError("Implement me, then re-run this cell")

    def get(self, version_id: str) -> PromptVersion:
        raise NotImplementedError("Implement me, then re-run this cell")


PromptRegistry = check("ch09-prompt-version", PromptRegistry)

In [3]:
registry = PromptRegistry()
registry.publish("v1", "You are a helpful support assistant. Answer the customer's question directly.")
registry.promote("v1")
print(f"current_version = {registry.current_version!r}")
print(f"history = {registry.history}")

current_version = 'v1'
history = ['v1']


### A deterministic response simulator

Standing in for real model calls (same real-vs-mock reasoning as every prior chapter, but
here the point isn't the model call itself, it's the deployment machinery around it), this
generates a synthetic but deterministic outcome per (prompt version, request), so this
chapter's canary/drift demos produce the same numbers every run.

In [4]:
import hashlib


def simulate_response(prompt_text: str, request_id: str) -> dict:
    '''Deterministic stand-in for a real model call. Two independent signals derived from a
    hash of (prompt text, request id): a refusal probability and a response-length range.
    Real behavior differences between prompt versions are simulated by prompt TEXT actually
    affecting these signals below -- not hardcoded per version_id -- so this genuinely reacts
    to what a prompt says, the same way a real model's behavior would.'''
    h = int(hashlib.sha256(f"{prompt_text}::{request_id}".encode()).hexdigest(), 16)

    base_refusal_rate = 0.03
    if "cautious" in prompt_text.lower() or "when in doubt, decline" in prompt_text.lower():
        base_refusal_rate = 0.35  # a prompt rewrite that overcorrects toward refusing

    refused = (h % 1000) / 1000 < base_refusal_rate
    length = 80 + (h % 150)
    return {"refused": refused, "length": length}


sample = simulate_response(registry.get("v1").text, "req-00001")
print(f"Sample response signal: {sample}")


Sample response signal: {'refused': False, 'length': 107}


### Canary routing

A sticky router: the same `request_id` always routes to the same version, so a given user
gets a consistent experience across a session instead of flip-flopping between prompt
versions call to call.

Getting the right *share* of traffic onto the canary is the easy half, and `random()` does it
perfectly. The half that matters is that the assignment has to be **stable**: the same request
lands on the same side every time, in every process, forever. Re-roll per call and a user
flips between prompt versions mid-conversation, your metrics mix the two populations
together, and no bug report is ever reproducible.

Hash the id and split on the hash. That also buys you the ramp property for free -- widening
the canary from 5% to 25% adds people to the cohort without moving anyone out of it.

In [ ]:
def route_request(request_id: str, stable_version: str, canary_version: str, canary_pct: float) -> str:
    '''Decide which prompt version this request sees.

    Return canary_version for canary_pct percent of requests and stable_version for the rest.
    The assignment must be a pure function of request_id -- hash it (hashlib.sha256 on the
    encoded id), take the hash modulo 100, and compare against canary_pct.

    Two properties fall out of doing it that way, and both matter:
      - the same request id always routes the same way, in any process, on any day
      - raising canary_pct only ever ADDS requests to the canary cohort, never moves any out

    canary_pct=0 means nobody, canary_pct=100 means everybody.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


route_request = check("ch09-canary-split", route_request)

In [6]:
registry.publish("v2", "You are a helpful support assistant. Answer the customer's question directly and concisely.")

request_ids = [f"req-{i:05d}" for i in range(1000)]
routed = [route_request(rid, "v1", "v2", canary_pct=10) for rid in request_ids]
canary_share = routed.count("v2") / len(routed) * 100
print(f"Canary share of traffic: {canary_share:.1f}% (target: 10%)")

Canary share of traffic: 9.3% (target: 10%)


### Shadow deployment

Run the canary version on the same requests without serving its output. Log both, serve only
the stable response, compare offline. This is strictly safer than a live canary for a change
you're not confident about yet.

In [7]:
def shadow_compare(request_ids: list, stable_text: str, shadow_text: str) -> dict:
    stable_results = [simulate_response(stable_text, rid) for rid in request_ids]
    shadow_results = [simulate_response(shadow_text, rid) for rid in request_ids]

    stable_refusal_rate = sum(r["refused"] for r in stable_results) / len(stable_results)
    shadow_refusal_rate = sum(r["refused"] for r in shadow_results) / len(shadow_results)

    return {
        "stable_refusal_rate": stable_refusal_rate,
        "shadow_refusal_rate": shadow_refusal_rate,
        "delta": shadow_refusal_rate - stable_refusal_rate,
        "served_to_users": "stable only -- shadow output never shown, this comparison is offline",
    }


comparison = shadow_compare(request_ids, registry.get("v1").text, registry.get("v2").text)
for k, v in comparison.items():
    print(f"{k}: {v}")


stable_refusal_rate: 0.036
shadow_refusal_rate: 0.04
delta: 0.0040000000000000036
served_to_users: stable only -- shadow output never shown, this comparison is offline


## Break it 1: an in-place prompt edit that makes "rollback" a no-op

This is a plain mutable store: exactly what "just quickly tweak the prompt in the config"
looks like in code, with no registry enforcing anything.

In [8]:
class NaivePromptStore:
    '''The bug: prompt text lives in a plain mutable dict, edited in place. No history, no
    immutability -- nothing stops the same key from silently pointing at different text
    over time.'''
    def __init__(self):
        self.prompts: dict[str, str] = {}

    def set(self, version_id: str, text: str) -> None:
        self.prompts[version_id] = text  # overwrites silently if version_id already exists


naive = NaivePromptStore()
naive.set("v1", "You are a helpful support assistant. Answer the customer's question directly.")
print("Before edit:", naive.prompts["v1"])

# Someone "quickly tweaks" the live prompt to be more conservative -- still under the same
# key, v1, because nobody thought of this as "publishing a new version."
naive.set("v1", "You are a cautious support assistant. When in doubt, decline to answer.")
print("After  edit:", naive.prompts["v1"])

print("\n'Rolling back to v1' does nothing -- v1 IS the edited, regressed text now:")
print(f"  {naive.prompts['v1']!r}")


Before edit: You are a helpful support assistant. Answer the customer's question directly.
After  edit: You are a cautious support assistant. When in doubt, decline to answer.

'Rolling back to v1' does nothing -- v1 IS the edited, regressed text now:
  'You are a cautious support assistant. When in doubt, decline to answer.'


Diagnose before reading on. What would you check to confirm this is what happened, and what
does the fix actually need to guarantee?

Write your diagnosis in the next cell before moving on. The worked answer is in `solutions/reference/ch09.py`; read it after yours passes, not before.

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_ROLLBACK_DIAGNOSIS = """Replace this with your diagnosis."""


MY_ROLLBACK_DIAGNOSIS = check("ch09-diagnose-mutable-prompt", MY_ROLLBACK_DIAGNOSIS)

In [10]:
registry.publish("v3-regressed", "You are a cautious support assistant. When in doubt, decline to answer.")
registry.promote("v3-regressed")
print(f"Live now: {registry.current_version} -> {registry.get(registry.current_version).text!r}")

# The regression is noticed -- roll back.
registry.promote("v1")
print(f"\nRolled back to: {registry.current_version}")
print(f"v1's text, completely unchanged by anything that happened to v3-regressed:")
print(f"  {registry.get('v1').text!r}")

print("\nAnd the mistake from the naive store above is now structurally impossible:")
try:
    registry.publish("v1", "some completely different text")
except ValueError as e:
    print(f"  Correctly rejected: {e}")


Live now: v3-regressed -> 'You are a cautious support assistant. When in doubt, decline to answer.'

Rolled back to: v1
v1's text, completely unchanged by anything that happened to v3-regressed:
  "You are a helpful support assistant. Answer the customer's question directly."

And the mistake from the naive store above is now structurally impossible:
  Correctly rejected: version 'v1' already exists -- versions are immutable, publish a new id


## Break it 2: a canary rollout that ships a real regression

An automated progressive rollout: start small, check health after each stage, ramp up if
healthy, halt and roll back if not. The regressed prompt from break-it 1
(`v3-regressed`, refusal rate ~35% vs. `v1`'s ~3%) goes through this exact rollout.

In [11]:
def check_canary_health(stable_rate: float, canary_rate: float, threshold: float = 0.5) -> bool:
    '''The bug: a drift-detection threshold set so loosely that it never actually catches
    anything. 0.5 (a 50-percentage-point swing) sounds like a "safe, conservative" number if
    you're not thinking in terms of what a real regression's magnitude actually looks like --
    but this chapter's real regression is "only" about 32 points, comfortably under it.'''
    return abs(canary_rate - stable_rate) < threshold


def run_progressive_rollout(stable_text: str, canary_text: str, health_check, stages=(5, 25, 50, 100)) -> dict:
    log = []
    for pct in stages:
        sample_ids = [f"rollout-req-{i:05d}" for i in range(500)]
        stable_results = [simulate_response(stable_text, rid) for rid in sample_ids]
        canary_results = [simulate_response(canary_text, rid) for rid in sample_ids]
        stable_rate = sum(r["refused"] for r in stable_results) / len(stable_results)
        canary_rate = sum(r["refused"] for r in canary_results) / len(canary_results)

        healthy = health_check(stable_rate, canary_rate)
        log.append({"stage_pct": pct, "stable_refusal_rate": stable_rate,
                     "canary_refusal_rate": canary_rate, "healthy": healthy})
        if not healthy:
            return {"outcome": "ROLLED BACK", "final_stage_pct": pct, "log": log}
    return {"outcome": "FULLY PROMOTED", "final_stage_pct": 100, "log": log}


result = run_progressive_rollout(registry.get("v1").text, registry.get("v3-regressed").text, check_canary_health)
print(f"Outcome: {result['outcome']}")
for entry in result["log"]:
    print(f"  {entry['stage_pct']:3d}% -- stable={entry['stable_refusal_rate']:.3f} "
          f"canary={entry['canary_refusal_rate']:.3f} healthy={entry['healthy']}")


Outcome: FULLY PROMOTED
    5% -- stable=0.036 canary=0.366 healthy=True
   25% -- stable=0.036 canary=0.366 healthy=True
   50% -- stable=0.036 canary=0.366 healthy=True
  100% -- stable=0.036 canary=0.366 healthy=True


Diagnose before reading on. The regressed prompt just shipped to 100% of traffic. What's
wrong with `check_canary_health`, specifically, not "the rollout process," but the function
itself?

Write your diagnosis in the next cell before moving on. The worked answer is in `solutions/reference/ch09.py`; read it after yours passes, not before.

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_THRESHOLD_DIAGNOSIS = """Replace this with your diagnosis."""


MY_THRESHOLD_DIAGNOSIS = check("ch09-diagnose-loose-threshold", MY_THRESHOLD_DIAGNOSIS)

Now write the calibrated version. The threshold is the entire content of this function, and
it is wrong in both directions if you are careless with it.

Too loose and it never fires -- 0.5 above is a round, conservative-*sounding* number that
only trips above a fifty-point swing, which no real regression ever reaches. Too tight, or
absent altogether, and it fires on every rollout, because two 500-request samples never
produce identical rates. A rollback signal that cries wolf gets muted inside a week, and a
muted signal is worth exactly as much as no signal.

In [ ]:
def check_canary_health_fixed(stable_rate: float, canary_rate: float, threshold: float = 0.05) -> bool:
    '''Is the canary's behaviour close enough to the stable version's to keep rolling?

    Return True when the two rates are within `threshold` of each other, False otherwise.
    Compare the absolute difference: a canary that refuses thirty points LESS often has also
    changed sharply, and an unexplained improvement is worth stopping a rollout for too.

    A difference of exactly the threshold is not inside it. The default of 0.05 is calibrated
    against what a real regression looks like -- five percentage points, not fifty -- so it
    has to catch this chapter's 32-point swing without anyone passing an explicit threshold.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


check_canary_health_fixed = check("ch09-drift-detect", check_canary_health_fixed)

In [14]:
result_fixed = run_progressive_rollout(registry.get("v1").text, registry.get("v3-regressed").text, check_canary_health_fixed)
print(f"Outcome: {result_fixed['outcome']}")
for entry in result_fixed["log"]:
    print(f"  {entry['stage_pct']:3d}% -- stable={entry['stable_refusal_rate']:.3f} "
          f"canary={entry['canary_refusal_rate']:.3f} healthy={entry['healthy']}")

Outcome: ROLLED BACK
    5% -- stable=0.036 canary=0.366 healthy=False


## Containerizing the agent

This repository has a real, working root-level `Dockerfile`; open it alongside this
notebook. It's not a snippet embedded here. It's a genuine, standalone artifact that
containerizes this repo's Python environment, built and reasoned about the same way the rest
of this course insists on: real code, not a description of what real code would look like.

A few choices are worth calling out explicitly, since they're the kind of thing a reviewer
actually checks in a real Dockerfile, not just "does it build." `requirements.txt` is copied
and installed before the source code, so editing a notebook doesn't invalidate the (usually
much slower) dependency-install layer in Docker's build cache; reversing this order is one
of the most common real Dockerfile mistakes. The container runs as `agent`, not `root`,
applying least privilege to the container boundary itself, the same instinct Chapter 6
applied to tool scoping. `LLM_PROVIDER` and API keys are supplied at `docker run` time via
environment variables and are never copied into a layer at build time, since a secret in a
Docker layer is recoverable from the image itself even if a later layer "removes" it; this
isn't just a style preference. And the image includes a `HEALTHCHECK`. It's cheap to add,
and it's the container-level analogue of this chapter's own theme: knowing a container is
running isn't the same as knowing it's healthy.

In [15]:
dockerfile_path = _repo_root / "Dockerfile"
dockerfile_text = dockerfile_path.read_text()
print(dockerfile_text)


# Containerizes this repository's Python environment so the curriculum's agent code (the
# capstone agent, or any individual chapter's notebook run headlessly) can run the same way
# regardless of the host machine. See curriculum/09_llmops_deployment.ipynb for the
# containerization concept section this file accompanies, and PROGRESS.md's Unit 10 notes
# for how this file was verified in this repo's own build environment.

FROM python:3.11-slim

# A non-root user to run the agent as -- least privilege inside the container too, not just
# in the agent's own tool scoping (Chapter 6's principle, applied one layer down).
RUN useradd --create-home --uid 1000 agent
WORKDIR /app

# Copy dependency manifests first so Docker's layer cache is only invalidated by a real
# dependency change, not by every source-code edit -- the single most common Dockerfile
# performance mistake, worth doing correctly even in a course example.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.

### A note on verifying this in the environment this course was built in

This repo's own build environment has a working Docker daemon and CLI, but its sandboxed
network policy blocks the CDN hosts Docker Hub and GitHub Container Registry actually serve
image layer blobs from (`production.cloudfront.docker.com`,
`pkg-containers.githubusercontent.com`). This was confirmed by directly attempting `docker
pull python:3.11-slim` and a GHCR image, both of which fail at the blob-download step
specifically, not at the registry-API step. A `FROM scratch` build with no external base
image completes instantly in the same environment, confirming this is genuinely a
network-policy block on those two CDNs, not a broader problem with Docker itself. This means
`docker build` on the Dockerfile above could not be executed end-to-end and verified in this
specific build session: a real, honestly-reported limitation, not something worked around
with a substitute (there's no alternate "real" source for a base container image the way
there was for SQuAD or `tiktoken`'s data in earlier chapters). Anyone running this in a
normal network environment should expect `docker build -t agent-interview-prep .` to work
correctly; see `PROGRESS.md`'s Unit 10 notes for the full verification trail.

## Recap

This chapter covered what's genuinely different about deploying LLM/agent systems versus
traditional software: the prompt itself as a versioned, deployable artifact, not just
configuration. Built an immutable `PromptRegistry` (publish, never overwrite), a sticky
canary router, and a shadow-deployment comparison, then broke two real things: an in-place
prompt edit that made "rollback" a structural no-op, and a canary-health threshold so loosely
calibrated that a genuine 32-point regression shipped to 100% of traffic without tripping a
single check. Closed with this repo's actual root `Dockerfile`, including an honest account
of what could and couldn't be verified end-to-end in this build's own sandboxed environment.

Next: with all nine chapters built, `interview_prep/` assembles the cross-chapter question
bank and mock-interview tooling this course has been pointing at since Chapter 1's
solutions-file convention, and the capstone puts everything from Chapters 1-9 into one
project.

## Interview drill

Answer each of these on your own before checking
`solutions/ch09_llmops_deployment_answers.md`.


1. Cold diagnosis. A prompt change shipped a week ago. This week, a stakeholder reports
"the assistant seems to be declining more requests than it used to." No infrastructure alert
ever fired. Walk through how you'd investigate, and why an infrastructure-health dashboard
alone wouldn't have caught this.

2. Design judgment. A teammate proposes skipping shadow deployment entirely and going
straight to a 5% canary for every prompt change, to "move faster." When would you push back,
and when would you agree it's fine?

3. Judgment call. You're picking a drift-detection threshold for refusal-rate monitoring
on a new deployment with no historical data yet. How would you actually arrive at a number,
rather than picking a round-sounding one?

4. Conceptual. What's the difference between "the model provider silently updated the
model behind a stable API name" and every other failure mode this chapter covers? What, if
anything, can you actually do about it?

#### Answering these

There is a slot below for each question. Write your answer into it, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side.

`check(n)` will not show you an answer until you have written one of your own — once you have
read the model answer you can no longer find out what you actually knew. If you want it
anyway, `drill.reveal(n)` is there and makes no judgement.

Answers are read from `solutions/ch09_*_answers.md` at runtime, so nothing in this
notebook contains one.

In [16]:
from agentlib.self_check import drill as open_drill

drill = open_drill(9)
drill.questions()

Chapter 9 written drill — 4 questions

1. Cold diagnosis: refusals climbed, no infrastructure alert fired
2. Design judgment: skip shadow deployment, go straight to a 5% canary
3. Judgment call: picking a threshold with no historical data
4. Conceptual: silent upstream model drift


In [17]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 9: 0/4 answered
  still open: [1, 2, 3, 4]


In [18]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  Cold diagnosis: refusals climbed, no infrastructure alert fired

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)
